In [0]:
# 05_orchestration/5_elt_pipeline.py
# Pipeline ELT completo (ejecución local)

print("="*60)
print("INICIANDO PIPELINE ELT COMPLETO")
print("="*60)

# =====================================================
# PASO 1: Verificar que los esquemas existen
# =====================================================

print("\n Verificando esquemas...")
spark.sql("CREATE SCHEMA IF NOT EXISTS capa_raw")
spark.sql("CREATE SCHEMA IF NOT EXISTS capa_bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS capa_silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS capa_gold")
print(" Esquemas verificados")

# =====================================================
# PASO 2: Ejecutar notebooks en orden
# =====================================================

# Lista de notebooks (usar rutas relativas)
notebooks = [
    ("01_raw/1_ingest_to_raw", "Ingesta a RAW"),
    ("02_bronze/2_bronze_transform", "Transformación a Bronze"),
    ("03_silver/3_silver_transform", "Limpieza a Silver"),
    ("04_gold/4_gold_transform", "Modelo Estrella Gold")
]

print("\n▶ Ejecutando pipeline...\n")

for i, (notebook, desc) in enumerate(notebooks, 1):
    print(f"[{i}/4] {desc}...")
    try:
        # Ejecutar notebook con timeout de 300 segundos (5 minutos)
        dbutils.notebook.run(notebook, 300)
        print(f"   {desc} completado")
    except Exception as e:
        print(f"   ERROR en {desc}: {e}")
        print(f"   Pipeline interrumpido en paso {i}")
        raise

# =====================================================
# PASO 3: Verificar resultados
# =====================================================

print("\n Verificando tablas creadas...")

tablas = {
    "capa_raw.coffee_sales_raw": "RAW",
    "capa_bronze.coffee_sales_bronze": "BRONZE",
    "capa_silver.coffee_sales_silver": "SILVER",
    "capa_gold.fact_sales": "GOLD (Hechos)",
    "capa_gold.dim_product": "GOLD (Producto)",
    "capa_gold.dim_date": "GOLD (Fecha)",
    "capa_gold.dim_time": "GOLD (Hora)",
    "capa_gold.dim_day": "GOLD (Día)",
    "capa_gold.dim_month": "GOLD (Mes)",
    "capa_gold.dim_season": "GOLD (Temporada)"
}

print("\n Resumen de tablas:")
for tabla, nombre in tablas.items():
    try:
        count = spark.table(tabla).count()
        print(f"   {nombre}: {count} registros")
    except:
        print(f"   {nombre}: NO ENCONTRADA")

print("\n" + "="*60)
print(" ¡PIPELINE ELT COMPLETADO EXITOSAMENTE!")
print(" Datos listos para Power BI")
print("="*60)

# Mostrar resumen final
print("\n RESUMEN FINAL:")
print(f"   Total de transacciones: {spark.table('capa_gold.fact_sales').count()}")
print(f"   Productos únicos: {spark.table('capa_gold.dim_product').count()}")
print(f"   Días únicos: {spark.table('capa_gold.dim_date').count()}")
print(f"   Temporadas: {spark.table('capa_gold.dim_season').count()}")